In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from google.colab import drive

# Mount Drive - images and CSV already saved here from CP2
drive.mount('/content/drive')

# Upload your labels_all.csv first before running
# These paths use what was already saved in CP2
CSV_FILE = "/content/labels_all.csv"
IMG_DIR  = "/content/drive/MyDrive/nutrition5k_all_resized"
SAVE_DIR = "/content/drive/MyDrive/nutrition_checkpoints_final"
os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}", flush=True)

class NutritionDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.data[['total_calories', 'total_fat',
                                  'total_carb', 'total_protein']].values
        self.filenames = self.data['dish_id'].values

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = self.filenames[idx] + "_rgb.png"
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label, self.filenames[idx]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Upload labels_all.csv to Colab before running this
full_df   = pd.read_csv(CSV_FILE)
demo_df   = full_df.iloc[:5]
remaining = full_df.iloc[5:].reset_index(drop=True)

train_size = int(0.8 * len(remaining))
train_df   = remaining.iloc[:train_size]
test_df    = remaining.iloc[train_size:]

print(f"Train: {len(train_df)}, Test: {len(test_df)}, Demo: {len(demo_df)}", flush=True)

train_dataset = NutritionDataset(train_df, IMG_DIR, transform=train_transform)
test_dataset  = NutritionDataset(test_df,  IMG_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

# Fine-tune all layers - new in final submission
# CP2 only trained the final layer by default
# Now entire ResNet-50 learns from food images
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = True
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)
model = model.to(device)
print("Model loaded with all layers unfrozen!", flush=True)

# Check for existing checkpoint and resume
checkpoint_path = None
for i in range(35, 0, -1):
    path = f"{SAVE_DIR}/nutrition_model_epoch{i}.pth"
    if os.path.exists(path):
        checkpoint_path = path
        start_epoch = i
        break

if checkpoint_path:
    model.load_state_dict(torch.load(checkpoint_path))
    print(f"Resumed from epoch {start_epoch}", flush=True)
else:
    start_epoch = 0
    print("Starting fresh training", flush=True)

criterion  = nn.MSELoss()
optimizer  = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler - new in final submission
# CP2 used fixed learning rate throughout all epochs
# This automatically reduces lr by half if loss does not
# improve for 3 consecutive epochs, preventing bouncing
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

def train_model(model, dataloader, criterion, optimizer, scheduler,
                total_epochs=35, start_epoch=0):
    model.train()
    for epoch in range(start_epoch, total_epochs):
        running_loss = 0.0
        for i, (inputs, labels, _) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            if i % 10 == 0:
                print(f"  Epoch {epoch+1}, Batch {i}/{len(dataloader)}", flush=True)

        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch+1}/{total_epochs}, Loss: {epoch_loss:.4f}", flush=True)

        scheduler.step(epoch_loss)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current LR: {current_lr}", flush=True)

        save_path = f"{SAVE_DIR}/nutrition_model_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f"Saved: {save_path}", flush=True)

def evaluate_model(model, dataloader):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels, _ in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_preds  = np.concatenate(all_preds,  axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    mse = np.mean((all_preds - all_labels) ** 2)
    mae = np.mean(np.abs(all_preds - all_labels), axis=0)

    print(f"\nTest MSE: {mse:.4f}", flush=True)
    print(f"MAE per nutrient:", flush=True)
    print(f"  Calories: {mae[0]:.2f}", flush=True)
    print(f"  Fat:      {mae[1]:.2f}g", flush=True)
    print(f"  Carbs:    {mae[2]:.2f}g", flush=True)
    print(f"  Protein:  {mae[3]:.2f}g", flush=True)

def predict_demo_images(model):
    model.eval()
    print("\nPredictions on held-out demo images (never seen by model):")
    print(f"{'Dish':<25} {'Cal(P)':<10} {'Cal(A)':<10} {'Fat(P)':<8} {'Fat(A)':<8} {'Carbs(P)':<10} {'Carbs(A)':<10} {'Pro(P)':<8} {'Pro(A)':<8}")
    print("-" * 100)
    for _, row in demo_df.iterrows():
        img_path = os.path.join(IMG_DIR, row['dish_id'] + "_rgb.png")
        image = Image.open(img_path).convert("RGB")
        image = test_transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(image).cpu().numpy()[0]
        print(f"{row['dish_id']:<25} "
              f"{pred[0]:<10.1f} {row['total_calories']:<10.1f} "
              f"{pred[1]:<8.1f} {row['total_fat']:<8.1f} "
              f"{pred[2]:<10.1f} {row['total_carb']:<10.1f} "
              f"{pred[3]:<8.1f} {row['total_protein']:<8.1f}")

def predict_one(model, dish_id):
    model.eval()
    img_path = os.path.join(IMG_DIR, dish_id + "_rgb.png")
    image = Image.open(img_path).convert("RGB")
    image = test_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(image).cpu().numpy()[0]
    actual = full_df[full_df['dish_id'] == dish_id].iloc[0]
    print(f"\nResults for {dish_id}:")
    print(f"{'Nutrient':<12} {'Predicted':>12} {'Actual':>12} {'Difference':>12}")
    print("-" * 50)
    print(f"{'Calories':<12} {pred[0]:>12.2f} {actual['total_calories']:>12.2f} {abs(pred[0]-actual['total_calories']):>12.2f}")
    print(f"{'Fat':<12} {pred[1]:>12.2f} {actual['total_fat']:>12.2f} {abs(pred[1]-actual['total_fat']):>12.2f}")
    print(f"{'Carbs':<12} {pred[2]:>12.2f} {actual['total_carb']:>12.2f} {abs(pred[2]-actual['total_carb']):>12.2f}")
    print(f"{'Protein':<12} {pred[3]:>12.2f} {actual['total_protein']:>12.2f} {abs(pred[3]-actual['total_protein']):>12.2f}")

# Run everything
train_model(model, train_loader, criterion, optimizer, scheduler,
            total_epochs=35, start_epoch=start_epoch)
evaluate_model(model, test_loader)
predict_demo_images(model)
predict_one(model, "dish_1561662216")

Mounted at /content/drive
Using device: cuda
Train: 2787, Test: 697, Demo: 5
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 175MB/s]


Model loaded with all layers unfrozen!
Starting fresh training
  Epoch 1, Batch 0/88
  Epoch 1, Batch 10/88
  Epoch 1, Batch 20/88
  Epoch 1, Batch 30/88
  Epoch 1, Batch 40/88
  Epoch 1, Batch 50/88
  Epoch 1, Batch 60/88
  Epoch 1, Batch 70/88
  Epoch 1, Batch 80/88
Epoch 1/35, Loss: 13779.9838
Current LR: 0.001
Saved: /content/drive/MyDrive/nutrition_checkpoints_final/nutrition_model_epoch1.pth
  Epoch 2, Batch 0/88
  Epoch 2, Batch 10/88
  Epoch 2, Batch 20/88
  Epoch 2, Batch 30/88
  Epoch 2, Batch 40/88
  Epoch 2, Batch 50/88
  Epoch 2, Batch 60/88
  Epoch 2, Batch 70/88
  Epoch 2, Batch 80/88
Epoch 2/35, Loss: 5387.1047
Current LR: 0.001
Saved: /content/drive/MyDrive/nutrition_checkpoints_final/nutrition_model_epoch2.pth
  Epoch 3, Batch 0/88
  Epoch 3, Batch 10/88
  Epoch 3, Batch 20/88
  Epoch 3, Batch 30/88
  Epoch 3, Batch 40/88
  Epoch 3, Batch 50/88
  Epoch 3, Batch 60/88
  Epoch 3, Batch 70/88
  Epoch 3, Batch 80/88
Epoch 3/35, Loss: 4834.4255
Current LR: 0.001
Saved: /co